<a href="https://colab.research.google.com/github/Nikibort/data_303_project/blob/main/data_303_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install folium
!pip install geopandas

In [ ]:
import pandas as pd
import folium
import geopandas as gpd
import folium

In [ ]:
data = pd.read_csv('san_antonio.csv', low_memory = False)
data.head()
#data.shape
#data['type'].unique()
#data['raw_custodial_arrest_made'].unique()

In [ ]:
#dropping raws with NaN in important columns
data.dropna(subset=['raw_custodial_arrest_made','raw_row_number', 'date', 'time', 'location', 'lat', 'lng', 'subject_race', 'type'], inplace=True)
data.head()
#data.shape


In [ ]:
data['date'] = pd.to_datetime(data['date'])
data_full = data.copy()
data = data[data['date'].dt.year == 2018]
data.shape
data.head()

In [ ]:
#dropping useless columns
data = data.drop(columns = ['location', 'raw_contraband_or_evidence','vehicle_make',	'vehicle_model', 'vehicle_registration_state', 'geocode_source', 'substation', 'vehicle_color', 'raw_posted_speed', 'raw_actual_speed'])

In [ ]:
#shortened dataset
data_short = data.iloc[0:4000]

In [ ]:
#found neighborhood demarcation online - its also in our github
neighborhoods_file = 'SAPD_Districts_1445405653252499739.geojson'

In [ ]:
neighborhoods = gpd.read_file(neighborhoods_file)
neighborhoods.head()

In [ ]:
data_short['geometry'] = gpd.points_from_xy(data_short.lng, data_short.lat)
data_short = gpd.GeoDataFrame(data_short, geometry='geometry')
data_short.head()

In [ ]:
m = folium.Map(location=[29.4241, -98.4936], zoom_start=11)
folium.GeoJson(neighborhoods, name="Neighborhoods", smooth_factor=2,
                    style_function=lambda x: {'color':'blue','fillColor':'transparent','weight':1.2},tooltip=folium.GeoJsonTooltip(fields=['SUBSTN'])).add_to(m)
folium.LayerControl().add_to(m)

m

In [ ]:
color_map = {
    "pedestrian": "red",
    "vehicular" : "green"

}

joined = gpd.sjoin(data_short, neighborhoods, how="left", predicate='within')
joined = joined.rename(columns={'type': 'stop_type'})
joined.head()

for i, row in joined.iterrows():

  color = color_map.get(row.stop_type, "gray")
  if row.raw_custodial_arrest_made == 'Yes':
      folium.Marker(
          location = [row.lat, row.lng],
          icon=folium.Icon(
                color='yellow',
                icon='star',
                prefix='fa'
            ),
          tooltip = f"""
          Stop Type: {row.stop_type}<br>
          Subject Age: {row.subject_age}<br>
          Subject Sex: {row.subject_sex}<br>
          Subject Race: {row.subject_race}<br>
          Arrested?: {row.raw_custodial_arrest_made}<br>
          """,
          popup = row.SUBSTN

      ).add_to(m)
  else:
      folium.CircleMarker(
          location = [row.lat, row.lng],
          radius = 4,
          color = color,
          fill = True,
          fill_color = color,
          fill_opacity = 0.7,
          popup = row.SUBSTN,
          tooltip = f"""
          Stop Type: {row.stop_type}<br>
          Subject Age: {row.subject_age}<br>
          Subject Sex: {row.subject_sex}<br>
          Subject Race: {row.subject_race}<br>
          Arrested?: {row.raw_custodial_arrest_made}<br>
          """
      ).add_to(m)


folium.LayerControl().add_to(m)


In [ ]:
print(joined['raw_custodial_arrest_made'].value_counts(dropna=False))

In [ ]:
m

In [ ]:
print(len(joined))

In [ ]:
from folium.plugins import HeatMap

heatmap = folium.Map(location = [data_full.lat.mean(), data_full.lng.mean()])
heat_coord = data_full[['lat', 'lng']]
HeatMap(heat_coord, radius = 10, blur = 10).add_to(heatmap)
heatmap
#